In [ ]:
#Activity 1
#Run the libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# 2. Load dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

In [ ]:
# 2. Basic data cleaning
# TotalCharges has blank values, so convert it to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows with missing TotalCharges
df = df.dropna()

print("Dataset shape after cleaning:", df.shape)

In [ ]:
# 3. Prepare features and target

# Drop customerID because it is only an identifier
X = df.drop(["customerID", "Churn"], axis=1)

# Convert target into numeric format
# No = 0, Yes = 1
y = df["Churn"].map({"No": 0, "Yes": 1})

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns

print("Categorical columns:", list(categorical_cols))
print("Numerical columns:", list(numeric_cols))

In [ ]:
# 4. Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
# 5. Preprocessing + Baseline SVM model

# Numerical columns are scaled
# Categorical columns are converted using one-hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# Baseline SVM uses default SVC settings
baseline_svm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC())
])

In [ ]:
# 6. Train baseline SVM model

baseline_svm.fit(X_train, y_train)

# Predict test data
y_pred = baseline_svm.predict(X_test)

In [ ]:
# 7. Evaluate baseline model

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("===== BASELINE SVM MODEL PERFORMANCE =====")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

In [ ]:
#check for overfitting
train_pred = baseline_svm.predict(X_train)
train_accuracy = accuracy_score(y_train, train_pred)
print(train_accuracy)